In [1]:
import os, random, time, json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score
from imblearn.metrics import specificity_score
from mambapy.vim import VMamba, MambaConfig
from thop import profile

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [2]:
AUG_ROOT = "D:/mamba_model/aug_clean_tio"
TAG      = "tio_concat"

COHORT_CSV = "D:/mamba_model/thesis_cohort_clean.csv"
MRI_CACHE  = f"{AUG_ROOT}/roi_mri"
PET_CACHE  = f"{AUG_ROOT}/roi_pet"
CKPT_DIR   = f"D:/mamba_model/checkpoints_v7_roi_{TAG}"
RESULTS    = f"D:/mamba_model/v7_roi_{TAG}_results.json"
PREFIX     = "v7_roi_concat"
os.makedirs(CKPT_DIR, exist_ok=True)

SPLIT_SEED  = 42
AUG_SEEDS   = [1, 101, 42]
BATCH_SIZE  = 4
NUM_WORKERS = 0

print(f"aug:  {AUG_ROOT}")
print(f"ckpt: {CKPT_DIR}")
for p in (MRI_CACHE, PET_CACHE):
    n = len(os.listdir(p)) if os.path.isdir(p) else 0
    print(f"  {os.path.basename(p)}: {n} files{'  *** MISSING ***' if n == 0 else ''}")

aug:  D:/mamba_model/aug_clean_tio
ckpt: D:/mamba_model/checkpoints_v7_roi_tio_concat
  roi_mri: 560 files
  roi_pet: 560 files


In [3]:
class VimEncoder(nn.Module):
    """Bidirectional Mamba over a token sequence."""
    def __init__(self, d_model=32, n_layers=2, d_state=16):
        super().__init__()
        cfg = MambaConfig(d_model=d_model, n_layers=n_layers, d_state=d_state,
                          bidirectional=True, divide_output=True,
                          pscan=True, use_cuda=False)
        self.encoder = VMamba(cfg)
        self.final_norm = nn.LayerNorm(d_model)
    def forward(self, tokens):
        return self.final_norm(self.encoder(tokens))

In [4]:
class ROIPatchEmbed3D(nn.Module):
    """6 ROIs -> non-overlapping 8^3 patches -> one token each. 3072 tokens."""
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32):
        super().__init__()
        self.n_rois, self.patch_size = n_rois, patch_size
        self.grid_size = roi_size // patch_size
        self.patches_per_roi = self.grid_size ** 3
        self.d_model = d_model
        self.patch_conv = nn.Conv3d(1, d_model, kernel_size=patch_size, stride=patch_size)
        self.roi_embed    = nn.Embedding(n_rois, d_model)
        self.depth_embed  = nn.Embedding(self.grid_size, d_model)
        self.height_embed = nn.Embedding(self.grid_size, d_model)
        self.width_embed  = nn.Embedding(self.grid_size, d_model)
        with torch.no_grad():
            for e in [self.roi_embed, self.depth_embed, self.height_embed, self.width_embed]:
                e.weight.mul_(0.02)
        d, h, w = torch.meshgrid(torch.arange(self.grid_size), torch.arange(self.grid_size),
                                 torch.arange(self.grid_size), indexing="ij")
        self.register_buffer("coordinates", torch.stack([d, h, w], -1).reshape(-1, 3),
                             persistent=False)

    def forward(self, rois):
        B, n = rois.shape[:2]
        x = rois.reshape(B * n, 1, *rois.shape[-3:])
        tokens = self.patch_conv(x).flatten(2).transpose(1, 2)
        tokens = tokens.reshape(B, n, self.patches_per_roi, self.d_model)
        c = self.coordinates
        spatial = (self.depth_embed(c[:, 0]) + self.height_embed(c[:, 1])
                   + self.width_embed(c[:, 2]))
        tokens = tokens + spatial[None, None] + self.roi_embed.weight[None, :, None, :]
        occ = F.max_pool3d((x.abs() > 1e-6).float(),
                           kernel_size=self.patch_size, stride=self.patch_size)
        valid = occ.flatten(1).bool().reshape(B, n, self.patches_per_roi)
        tokens = tokens.reshape(B, -1, self.d_model)
        valid = valid.reshape(B, -1)
        return tokens * valid.unsqueeze(-1).to(tokens.dtype), valid


class ROIBranch(nn.Module):
    """Patch embed -> Mamba -> average within each region."""
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32,
                 n_layers=2, d_state=16):
        super().__init__()
        self.n_rois = n_rois
        self.patch_embed = ROIPatchEmbed3D(n_rois, roi_size, patch_size, d_model)
        self.vim = VimEncoder(d_model, n_layers, d_state)

    def forward(self, rois):
        tokens, valid = self.patch_embed(rois)
        tokens = self.vim(tokens)
        B = tokens.shape[0]
        t = tokens.reshape(B, self.n_rois, -1, tokens.shape[-1])
        v = valid.reshape(B, self.n_rois, -1, 1).to(tokens.dtype)
        return (t * v).sum(2) / v.sum(2).clamp_min(1.0)          # (B, 6, d)


class ConcatOnlyModel(nn.Module):
    """Each region's MRI and PET summaries are concatenated
    directly.."""
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32,
                 n_layers=2, n_classes=2, d_state=16, dropout=0.4, **_):
        super().__init__()
        self.n_rois = n_rois
        self.mri_branch = ROIBranch(n_rois, roi_size, patch_size, d_model, n_layers, d_state)
        self.pet_branch = ROIBranch(n_rois, roi_size, patch_size, d_model, n_layers, d_state)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model * 2 * n_rois, n_classes)   # 384 -> 2

    def forward(self, mri_rois, pet_rois):
        mri_r = self.dropout(self.mri_branch(mri_rois))   # (B, 6, 32)
        pet_r = self.dropout(self.pet_branch(pet_rois))   # (B, 6, 32)
        B = mri_r.shape[0]
        fused = torch.cat([mri_r, pet_r], dim=2)          # (B, 6, 64)
        return self.classifier(self.dropout(fused.reshape(B, -1)))


In [5]:
df = pd.read_csv(COHORT_CSV)
sessions, labels = df["mri_session"].values, df["outcome_label"].values

X_tv, X_test, y_tv, y_test = train_test_split(
    sessions, labels, test_size=0.2, random_state=SPLIT_SEED, stratify=labels)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.25, random_state=SPLIT_SEED, stratify=y_tv)

session_to_subject = dict(zip(df["mri_session"], df["subject_id"]))
print(f"train {len(X_train)} | val {len(X_val)} | test {len(X_test)} "
      f"| test pos {int(y_test.sum())}")


_CACHE = {}

def load_cached(path):
    a = _CACHE.get(path)
    if a is None:
        a = np.load(path).astype(np.float32)
        _CACHE[path] = a
    return a


class MultimodalROIDataset(Dataset):
    def __init__(self, sessions, labels, mri_dir, pet_dir, is_train=False):
        self.samples, self.mri_dir, self.pet_dir = [], mri_dir, pet_dir
        for ses, lab in zip(sessions, labels):
            sid = session_to_subject[ses]
            self.samples.append((ses, sid, lab, "orig"))
            if is_train:
                for s in AUG_SEEDS:
                    self.samples.append((ses, sid, lab, f"aug{s}"))
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        mk, pk, lab, ver = self.samples[i]
        m = load_cached(f"{self.mri_dir}/{mk}_{ver}.npy")
        p = load_cached(f"{self.pet_dir}/{pk}_{ver}.npy")
        return (torch.from_numpy(m).unsqueeze(1), torch.from_numpy(p).unsqueeze(1),
                torch.tensor(lab, dtype=torch.long), mk)


def dl(ds, shuffle=False):
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=True,
                      persistent_workers=(NUM_WORKERS > 0))

mm_loaders = (dl(MultimodalROIDataset(X_train, y_train, MRI_CACHE, PET_CACHE, True), True),
              dl(MultimodalROIDataset(X_val,   y_val,   MRI_CACHE, PET_CACHE, False)),
              dl(MultimodalROIDataset(X_test,  y_test,  MRI_CACHE, PET_CACHE, False)))

print(f"train samples (4x augmented): {len(mm_loaders[0].dataset)}")

t0 = time.time()
for i, _ in enumerate(mm_loaders[0]):
    if i >= 20: break
cold = time.time() - t0
t0 = time.time()
for i, _ in enumerate(mm_loaders[0]):
    if i >= 20: break
print(f"20 batches: cold {cold:.1f}s -> warm {time.time()-t0:.1f}s")

train 120 | val 40 | test 40 | test pos 20
train samples (4x augmented): 480
20 batches: cold 13.8s -> warm 11.1s


In [6]:
def train_epoch(model, loader, opt, crit):
    model.train(); tot = 0
    for a, b, lb, _ in loader:
        opt.zero_grad()
        loss = crit(model(a.to(device), b.to(device)), lb.to(device))
        loss.backward(); opt.step(); tot += loss.item()
    return tot / len(loader)


def evaluate(model, loader, crit):
    model.eval(); tot, P, L = 0, [], []
    with torch.no_grad():
        for a, b, lb, _ in loader:
            out = model(a.to(device), b.to(device))
            tot += crit(out, lb.to(device)).item()
            P.extend(out.argmax(1).cpu().numpy()); L.extend(lb.numpy())
    return (tot/len(loader), np.mean(np.array(P) == np.array(L)),
            recall_score(L, P, zero_division=0), specificity_score(L, P))


def measure_inference(model, loader, n=20):
    model.eval(); ts = []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= n: break
            a, b = batch[0].to(device), batch[1].to(device); bs = a.shape[0]
            if device.type == 'cuda': torch.cuda.synchronize()
            t0 = time.time(); _ = model(a, b)
            if device.type == 'cuda': torch.cuda.synchronize()
            ts.append((time.time() - t0) / bs)
    return np.mean(ts), np.std(ts)


def compute_flops(model, loader):
    try:
        model.eval(); b = next(iter(loader))
        with torch.no_grad():
            macs, _ = profile(model, inputs=(b[0][:1].to(device), b[1][:1].to(device)),
                              verbose=False)
        return macs * 2
    except Exception as e:
        print(f"  (FLOPs failed: {e})"); return None


def run_seed(seed, model_cls, loaders, prefix,
             max_epochs=101, patience=15, min_epochs=25, lr=1e-4, log_every=5):
    torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    np.random.seed(seed); random.seed(seed)
    tr, va, te = loaders

    model = model_cls(d_model=32, n_layers=2, n_classes=2, dropout=0.4).to(device)
    crit  = nn.CrossEntropyLoss(label_smoothing=0.05)
    opt   = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    sch   = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=10)

    best, no_imp, best_ep, total = float('inf'), 0, 0, 0
    path = f"{CKPT_DIR}/{prefix}_seed{seed}.pt"
    print(f"\n--- {prefix} seed {seed} ---")

    for ep in range(1, max_epochs):
        t0 = time.time()
        trl = train_epoch(model, tr, opt, crit)
        vl, vacc, vtpr, vtnr = evaluate(model, va, crit)
        sch.step(vl); dt = time.time() - t0; total += dt
        if ep % log_every == 0 or ep == 1:
            print(f"  ep {ep:>3} | train {trl:.4f} | val {vl:.4f} | "
                  f"acc {vacc:.3f} tpr {vtpr:.3f} tnr {vtnr:.3f} | {dt:.0f}s")
        if vl < best:
            best, best_ep, no_imp = vl, ep, 0
            torch.save(model.state_dict(), path)
        else:
            no_imp += 1
            if ep >= min_epochs and no_imp >= patience:
                print(f"  early stop {ep}, best {best_ep}"); break

    if best_ep < 5:
        print(f"  WARNING: best epoch {best_ep} -- may not have trained")

    model.load_state_dict(torch.load(path, weights_only=True))
    _, acc, tpr, tnr = evaluate(model, te, crit)
    npar = sum(p.numel() for p in model.parameters() if p.requires_grad)
    inf_m, inf_s = measure_inference(model, te)
    fl = compute_flops(model, te)

    print(f"  >>> TEST Acc={acc*100:.1f}% TPR={tpr*100:.1f}% TNR={tnr*100:.1f}% | "
          f"params={npar:,} train={total/60:.1f}min inf={inf_m*1000:.2f}ms "
          f"{f'{fl/1e9:.2f}GFLOPs' if fl else ''} best_ep={best_ep}")

    return {"seed": seed, "acc": acc, "tpr": tpr, "tnr": tnr, "best_epoch": best_ep,
            "train_time_sec": total, "n_params": npar,
            "inf_time_ms": inf_m*1000, "inf_std_ms": inf_s*1000, "flops": fl}


results = {"mm": []}
print("ready")

ready


In [7]:
results["mm"].append(run_seed(1, ConcatOnlyModel, mm_loaders, f"{PREFIX}_mm"))


--- v7_roi_concat_mm seed 1 ---
  ep   1 | train 0.7026 | val 0.7046 | acc 0.500 tpr 1.000 tnr 0.000 | 70s
  ep   5 | train 0.6579 | val 0.6483 | acc 0.725 tpr 0.500 tnr 0.950 | 15s
  ep  10 | train 0.6300 | val 0.6254 | acc 0.675 tpr 0.500 tnr 0.850 | 16s
  ep  15 | train 0.6117 | val 0.6107 | acc 0.700 tpr 0.600 tnr 0.800 | 15s
  ep  20 | train 0.5690 | val 0.5931 | acc 0.750 tpr 0.750 tnr 0.750 | 15s
  ep  25 | train 0.5157 | val 0.5687 | acc 0.725 tpr 0.700 tnr 0.750 | 15s
  ep  30 | train 0.4381 | val 0.5463 | acc 0.775 tpr 0.700 tnr 0.850 | 16s
  ep  35 | train 0.3609 | val 0.5350 | acc 0.775 tpr 0.700 tnr 0.850 | 15s
  ep  40 | train 0.2694 | val 0.5189 | acc 0.750 tpr 0.700 tnr 0.800 | 24s
  ep  45 | train 0.2111 | val 0.5420 | acc 0.725 tpr 0.600 tnr 0.850 | 34s
  ep  50 | train 0.1601 | val 0.5693 | acc 0.725 tpr 0.500 tnr 0.950 | 33s
  ep  55 | train 0.1397 | val 0.5643 | acc 0.725 tpr 0.550 tnr 0.900 | 34s
  early stop 57, best 42
  >>> TEST Acc=72.5% TPR=75.0% TNR=70.0% |

In [8]:
results["mm"].append(run_seed(7, ConcatOnlyModel, mm_loaders, f"{PREFIX}_mm"))


--- v7_roi_concat_mm seed 7 ---
  ep   1 | train 0.7027 | val 0.6741 | acc 0.700 tpr 0.550 tnr 0.850 | 33s
  ep   5 | train 0.6699 | val 0.6382 | acc 0.725 tpr 0.550 tnr 0.900 | 34s
  ep  10 | train 0.6389 | val 0.6158 | acc 0.750 tpr 0.600 tnr 0.900 | 32s
  ep  15 | train 0.6025 | val 0.5974 | acc 0.750 tpr 0.550 tnr 0.950 | 34s
  ep  20 | train 0.5792 | val 0.5759 | acc 0.775 tpr 0.650 tnr 0.900 | 33s
  ep  25 | train 0.5160 | val 0.5571 | acc 0.825 tpr 0.750 tnr 0.900 | 33s
  ep  30 | train 0.4579 | val 0.5438 | acc 0.800 tpr 0.700 tnr 0.900 | 34s
  ep  35 | train 0.3770 | val 0.5349 | acc 0.775 tpr 0.600 tnr 0.950 | 36s
  ep  40 | train 0.2964 | val 0.5140 | acc 0.800 tpr 0.650 tnr 0.950 | 33s
  ep  45 | train 0.2267 | val 0.5008 | acc 0.775 tpr 0.650 tnr 0.900 | 36s
  ep  50 | train 0.1698 | val 0.5040 | acc 0.750 tpr 0.650 tnr 0.850 | 32s
  ep  55 | train 0.1421 | val 0.5640 | acc 0.750 tpr 0.600 tnr 0.900 | 35s
  ep  60 | train 0.1314 | val 0.5807 | acc 0.750 tpr 0.600 tnr 0.90

In [9]:
results["mm"].append(run_seed(123, ConcatOnlyModel, mm_loaders, f"{PREFIX}_mm"))


--- v7_roi_concat_mm seed 123 ---
  ep   1 | train 0.6996 | val 0.6750 | acc 0.675 tpr 0.600 tnr 0.750 | 33s
  ep   5 | train 0.6562 | val 0.6405 | acc 0.725 tpr 0.500 tnr 0.950 | 35s
  ep  10 | train 0.6272 | val 0.6173 | acc 0.725 tpr 0.500 tnr 0.950 | 37s
  ep  15 | train 0.5907 | val 0.5997 | acc 0.725 tpr 0.500 tnr 0.950 | 34s
  ep  20 | train 0.5738 | val 0.5729 | acc 0.775 tpr 0.700 tnr 0.850 | 33s
  ep  25 | train 0.5293 | val 0.5665 | acc 0.800 tpr 0.650 tnr 0.950 | 33s
  ep  30 | train 0.4586 | val 0.5454 | acc 0.775 tpr 0.600 tnr 0.950 | 32s
  ep  35 | train 0.3684 | val 0.5245 | acc 0.800 tpr 0.700 tnr 0.900 | 34s
  ep  40 | train 0.2935 | val 0.5397 | acc 0.775 tpr 0.650 tnr 0.900 | 35s
  ep  45 | train 0.2219 | val 0.5745 | acc 0.750 tpr 0.550 tnr 0.950 | 33s
  ep  50 | train 0.1717 | val 0.5639 | acc 0.725 tpr 0.600 tnr 0.850 | 33s
  early stop 50, best 35
  >>> TEST Acc=72.5% TPR=75.0% TNR=70.0% | params=90,562 train=28.7min inf=17.56ms 0.47GFLOPs best_ep=35


In [11]:
INCLUDE_SEEDS = [1, 7, 123]

def summarize(rs, name, include=INCLUDE_SEEDS):
    rs = [r for r in rs if r['seed'] in include]
    if not rs: print(f"{name}: no runs"); return
    a = [r['acc'] for r in rs]; t = [r['tpr'] for r in rs]; n = [r['tnr'] for r in rs]
    tm = [r['train_time_sec'] for r in rs]; inf = [r['inf_time_ms'] for r in rs]
    sd = (lambda v: np.std(v, ddof=1)*100 if len(v) > 1 else 0.0)
    print(f"{name}: Acc={np.mean(a)*100:.1f}±{sd(a):.1f}% "
          f"(range {min(a)*100:.1f}-{max(a)*100:.1f}) | "
          f"TPR={np.mean(t)*100:.1f}±{sd(t):.1f}% | TNR={np.mean(n)*100:.1f}±{sd(n):.1f}% | "
          f"Params={rs[0]['n_params']:,} | Train={np.mean(tm)/60:.1f}m | "
          f"Inf={np.mean(inf):.2f}ms | seeds={[r['seed'] for r in rs]}")
    print("    per seed: " + ", ".join(
        f"seed {r['seed']} {r['acc']*100:.1f}% (TPR {r['tpr']*100:.1f} / "
        f"TNR {r['tnr']*100:.1f}, best_ep {r['best_epoch']})" for r in rs))

print(f"=== v7 late per-region CONCATENATION — {TAG}, 200-subject cohort ===")
summarize(results['mm'], 'Multimodal')


with open(RESULTS, 'w') as f:
    json.dump({k: [{kk: (float(vv) if isinstance(vv, (float, np.floating)) else vv)
                    for kk, vv in r.items()} for r in v] for k, v in results.items()},
              f, indent=2)
print(f"\nsaved {RESULTS}")

=== v7 late per-region CONCATENATION — tio_concat, 200-subject cohort ===
Multimodal: Acc=70.8±2.9% (range 67.5-72.5) | TPR=75.0±0.0% | TNR=66.7±5.8% | Params=90,562 | Train=27.9m | Inf=19.72ms | seeds=[1, 7, 123]
    per seed: seed 1 72.5% (TPR 75.0 / TNR 70.0, best_ep 42), seed 7 67.5% (TPR 75.0 / TNR 60.0, best_ep 45), seed 123 72.5% (TPR 75.0 / TNR 70.0, best_ep 35)

saved D:/mamba_model/v7_roi_tio_concat_results.json


In [12]:
#  GFLOPs for one forward pass, batch size 1 -- late per-region concatenation

#  Architecture alone determines these figures, so no training, checkpoints
#  or data are needed.
import copy
from torch.utils.flop_counter import FlopCounterMode
from mambapy.vim import VMamba as _VMamba


def count_flops(model, inputs):
    m = copy.deepcopy(model).eval().cpu()
    inputs = [t.detach().cpu() for t in inputs]
    mamba_log, handles = [], []

    def mk(mod):
        def hook(_, inp, __):
            c = mod.config
            mamba_log.append(dict(L=inp[0].shape[1], ed=c.d_inner, n=c.d_state,
                                  layers=c.n_layers,
                                  bi=getattr(c, "bidirectional", False)))
        return hook
    for mod in m.modules():
        if isinstance(mod, _VMamba):
            handles.append(mod.register_forward_hook(mk(mod)))

    counter = FlopCounterMode(display=False)
    with torch.no_grad(), counter:
        m(*inputs)
    for h in handles:
        h.remove()

    cl = counter.get_total_flops()
    sc = sum(6 * r["ed"] * r["n"] * r["L"] * r["layers"] * (2 if r["bi"] else 1)
             for r in mamba_log)
    tokens = mamba_log[0]["L"] if mamba_log else 0
    del m
    return cl, sc, tokens


roi = lambda: torch.randn(1, 6, 1, 64, 64, 64)
model = ConcatOnlyModel().cpu()
cl, sc, tok = count_flops(model, [roi(), roi()])
p = sum(q.numel() for q in model.parameters() if q.requires_grad)

print("GFLOPs, one forward pass, batch size 1 -- late per-region concatenation\n")
print(f"  params              {p:,}")
print(f"  tokens per modality {tok:,}\n")
print(f"  conv + linear       {cl/1e9:.4f}")
print(f"  Mamba scan          {sc/1e9:.4f}")
print(f"  {'-'*32}")
print(f"  total               {(cl+sc)/1e9:.4f} GFLOPs")

GFLOPs, one forward pass, batch size 1 -- late per-region concatenation

  params              90,562
  tokens per modality 3,072

  conv + linear       0.5285
  Mamba scan          0.1510
  --------------------------------
  total               0.6795 GFLOPs
